# 011 — Ética desde el diseño y límites de automatización

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Ética desde el diseño:** los valores se implementan en decisiones técnicas concretas —
qué métrica se optimiza, qué datos entran, dónde se pone el umbral, qué se registra. No
elegir conscientemente no elimina la postura ética: la deja sin dueño.

- **Sesgo:** error sistemático desigual entre subgrupos; se *mide* por subgrupo (la
  accuracy global puede ocultar disparidades severas). Quitar la variable protegida no
  basta: las correlacionadas la reconstruyen (*proxy discrimination*).
- **Equidad:** múltiples definiciones formales (paridad demográfica, igualdad de
  oportunidades, calibración por grupo) **incompatibles entre sí** en general (Kleinberg
  et al.): elegir una es decisión de política documentable.
- **Transparencia:** *Model Cards* (Mitchell 2019) y *Datasheets* (Gebru 2021) — uso
  previsto, poblaciones evaluadas, límites.
- **Límites de automatización — 5 ejes:** reversibilidad del daño, asimetría de costos,
  vulnerabilidad de la población, explicabilidad exigible, deriva del entorno. A mayor
  puntuación, más cerca de "el sistema solo informa" y más lejos de "decide y ejecuta"
  (gradación que codifican el AI Act y el NIST AI RMF).

Ejemplo trabajado: mismo umbral global, misma tasa base de impago (10 %) → grupo B sufre el
doble de rechazos injustos (FPR 0.20 vs 0.10) y la mitad de recall (0.50 vs 0.80). La
accuracy global (0.83) no lo muestra; la tabla por subgrupo sí.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** A: recall 0.80, FPR 40/400 = 0.10, accuracy (80+360)/500 = 0.88.
B: recall 0.60, FPR 0.20, accuracy (60+320)/500 = 0.76. Global: (440+380)/1000 = **0.82**.
La global promedia y esconde que B recibe el doble de daño injusto con la misma tasa base
— el argumento central de medir por subgrupo.

**Ejercicio 2.** (a) Reversible, costo simétrico bajo, población no vulnerable →
automatización completa con monitoreo. (b) Daño poco reversible (oportunidad perdida),
población potencialmente vulnerable, explicabilidad exigible por ley en varias
jurisdicciones → el sistema prioriza/informa, humano decide con poder real. (c) Riesgo
vital, asimetría extrema → el sistema solo ordena la cola y SIEMPRE con revisión clínica;
partes pueden ser líneas rojas regulatorias.

**Ejercicio 3.** Colgar/transferir llamadas difíciles y responder rápido sin resolver
(genera rellamadas que ni siquiera cuentan igual). Métrica compuesta: resolución al primer
contacto confirmada + tasa de rellamada a 7 días + satisfacción muestral — más cara de
medir y más difícil de hackear, que es el trade-off típico.

**Ejercicio 4.** Bajar el umbral: con FN 10× más caro, conviene aceptar más falsos
positivos para reducir falsos negativos. Formalmente se minimiza el riesgo esperado
`10·P(FN) + 1·P(FP)`; el umbral óptimo se desplaza hacia clasificar como positivo ante
menos evidencia. El umbral es una decisión ética/económica explícita, no un default.

In [ ]:
result = run_lab("safety", seed=11)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — métricas por subgrupo
TP_A, FP_A, P_A, N_A = 80, 40, 100, 400
TP_B, FP_B, P_B, N_B = 60, 80, 100, 400
recall_A, fpr_A = TP_A / P_A, FP_A / N_A
recall_B, fpr_B = TP_B / P_B, FP_B / N_B
acc_A = (TP_A + (N_A - FP_A)) / 500
acc_B = (TP_B + (N_B - FP_B)) / 500
acc_global = (acc_A * 500 + acc_B * 500) / 1000
print(f"A: recall={recall_A} FPR={fpr_A} acc={acc_A}")
print(f"B: recall={recall_B} FPR={fpr_B} acc={acc_B}")
print(f"global acc={acc_global}  ← no revela la disparidad 2x en FPR")

In [ ]:
# Ejercicio 4 — umbral óptimo con costos asimétricos (ilustración numérica)
# riesgo(umbral) = 10 * P(FN) + 1 * P(FP), sobre puntuaciones sintéticas
import random
rng = random.Random(1)
positivos = [min(1, max(0, rng.gauss(0.7, 0.15))) for _ in range(1000)]
negativos = [min(1, max(0, rng.gauss(0.3, 0.15))) for _ in range(1000)]
for umbral in (0.3, 0.4, 0.5, 0.6):
    fn = sum(1 for s in positivos if s < umbral) / 1000
    fp = sum(1 for s in negativos if s >= umbral) / 1000
    print(f"umbral={umbral}: riesgo = 10*{fn:.3f} + {fp:.3f} = {10*fn + fp:.3f}")
# El mínimo cae por debajo de 0.5: el costo asimétrico empuja el umbral hacia abajo.

## Reflexión

1. En el JSON del laboratorio `safety`, ¿qué campo funciona como "model card embrionaria" y
   qué le faltaría para cumplir la documentación mínima de Mitchell et al. (2019)?
2. Toma una decisión automatizada que te afecte personalmente (crédito, spam, moderación,
   recomendación) y puntúala en los 5 ejes. ¿El grado de autonomía que tiene hoy coincide
   con el que tu análisis recomienda?
3. ¿Por qué "un humano revisa cada decisión" puede ser teatro de supervisión? ¿Qué métrica
   concreta revelaría si la supervisión es real?